In [ ]:
#import API
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("gemini")


In [ ]:
# ---  API Key Setup and Environment Configuration ---
import os
from kaggle_secrets import UserSecretsClient
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("gemini")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
except Exception as e:
    print(
        f"🔑 Authentication Error{e}"
    )


In [ ]:
# --- Agent Development Kit (ADK) Component Imports ---
from google.adk.agents import Agent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService


In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5, 
    exp_base=7,  
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], 
)

In [ ]:
memory_service = (
    InMemoryMemoryService()
    # Initializes `InMemoryMemoryService`. 
) 
session_service=InMemorySessionService()


In [ ]:
agent1=Agent(
    name="Agent1",
    
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
        # Uses the efficient 'flash-lite' model, which is ideal for structured 
    ),
    description="Validates and cleans user topics.",
    # Concise summary of the agent's singular, focused responsibility.
    instruction="""
You are Agent 1: Topic Validator.

Your job:

1. Clean the user’s topic.
   - Fix grammar
   - Make it 3–6 words
   - Remove slang

2. Decide if the topic is researchable.
   A topic is valid only if:
   - It is a clear subject (e.g., “Data Visualization using Python”)
   - It is safe (no hacking/illegal/harmful)
   - It is not too vague (“tell me something”, “help me”)
   - It is not personal info


Strict Rules:
- Do NOT provide research.
- Do NOT rewrite instructions.
- Do NOT add extra text.
Output MUST be a single JSON object with the following keys:
- "cleaned_topic": The 3–6 word cleaned topic OR the original topic if invalid.
- "pipeline_action": "CONTINUE" if valid, or "STOP" if invalid/unsafe.
""",
    tools=[],
    # Correctly has no tools, as its task is internal validation, not external search.
    output_key="validate",
    # Defines the key under which this agent's structured output will be stored 
    # in the runner's context for access by Agent 2.

)




In [ ]:
agent2= Agent(
    name="Agent2",
    # Clear functional name
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="Researches the cleaned topic.",
    instruction="""
You are Agent 2: Research Agent.

Task:
Given a cleaned topic, perform deep research using web search, websites, documentation, blogs, and trusted sources.

Rules:
1. Only include information from the last 24 months (2 years).
2. Do NOT create a roadmap.
3. Do NOT explain your process.
4. Extract and summarize only factual information.
5. Pass the information only
...
Strict Rules:
- Output MUST be a single JSON object containing all the defined research fields.
- No opinions, no roadmap, no chit chat, no pre-amble.

Definitions:
- key_concepts = fundamental ideas required to understand the topic.
- recent_trends = new developments from the last 2 years.
- important_tools = libraries, frameworks, software used for the topic.
- best_practices = updated, proven methods.
- recommended_resources = recent blogs, docs, courses (with clickable URLs).

Strict Rules:
- No opinions.
- No roadmap.
- No chit chat.
IMPORTANT SAFETY RULE:
- If the input is not valid research JSON from Agent 1, or if the topic is unsafe, harmful, illegal, or rejected by Agent 1, you MUST output exactly:
"INVALID_REQUEST"

Do NOT attempt to be helpful.
Do NOT generate educational content.
Just output: INVALID_REQUEST
""",
    # **Core Logic:** Instructs the agent to perform **deep research** # and **strict filtering** (e.g., "last 24 months").
    # **Security/Flow Control:** Includes a strong safety check (Output: "INVALID_REQUEST") 
    tools=[google_search],
    output_key="resource",
    
    #**Crucial:** Includes the `Google Search` tool, which is necessary for 
    
)


In [ ]:
agent3= Agent(
    name="Agent3",
    # Clear functional name.
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config,
        session_service=session_service,
        memory_service=memory_service,
    ),
    description="Researches the cleaned topic.",
    instruction="""
You are Agent 3: Roadmap Builder.

Your task:
Take the research JSON provided by Agent 2 and convert it into a clear, actionable learning roadmap.

Output format:
- A numbered list (15–20 steps)
- Sequential from absolute beginner → advanced
- Each step: short, practical, actionable
- Include tools, milestones, and mini-projects
- Provide important links at the end in a structured format
- Do NOT include unnecessary explanations

Rules:
- Do NOT repeat the research JSON.
- Do NOT restate the prompt.
- Do NOT output anything except the roadmap.
- Keep steps concise but specific.

Structure required:
1. Basics
2. Foundations
3. Hands-on skills
4. Tools & libraries
5. Intermediate concepts
6. Projects
7. Advanced topics
8. Final capstone or specialization

IMPORTANT SAFETY RULE:
- If the input is not valid research JSON from Agent 2, or if the topic is unsafe, harmful, illegal, or rejected by Agent 1, you MUST output exactly:
"INVALID_REQUEST"

Do NOT create a roadmap.
Do NOT attempt to be helpful.
Do NOT generate educational content.
Just output: INVALID_REQUEST

Otherwise (valid research JSON only):
- Produce a 15–20 step roadmap.
- Output ONLY the numbered steps.
- No extra text.

""",
    # **Core Logic:** Instructs the agent to synthesize the research into a 
    # **15–20 step, beginner-to-advanced, actionable roadmap**.
    tools=[],
    output_key="roadmap",
    # Correctly has no tools, as its task is pure synthesis and restructuring of 
    # internal (Agent 2) data.
    
)



In [ ]:
def stop_if_invalid(context):
    # context["current_agent_output"] will contain agent1 output
    output = context["validate"]

    if isinstance(output, str):
        
        import json
        output = json.loads(output)

    if "pipeline_action" in output and output["pipeline_action"] == "STOP":
        return True   # Stop sequence
        

    return False
    # If the action is "CONTINUE" or the key is missing (defaulting to continue), 
    


In [ ]:
root_agent = SequentialAgent(
    name="RoadmapMaker",
   
    sub_agents=[agent1, agent2, agent3],
    early_stopping_condition=stop_if_invalid,  
    session_service=session_service,
    memory_service=memory_service
     
    # 1. Agent 1 (Validator) → 2. Agent 2 (Researcher) → 3. Agent 3 (Builder).
)

In [ ]:

runner = InMemoryRunner(
    agent=root_agent,
    session_service=session_service,
    memory_service=memory_service
)


def generate_roadmap(user_topic: str):
    try:
        result = runner.run(
            agent_name="RoadmapMaker",
            user_message=user_topic,
            session_id="unique_session_id"  
        )
        
        
        final_output = result.get("roadmap", "")
        
        if final_output == "INVALID_REQUEST":
            return "❌ Invalid or unsafe topic. Please try a different subject."
        
        return final_output
        
    except Exception as e:
        return f"⚠️ Pipeline error: {str(e)}"


topic = "Data Visualization using Python"
roadmap = generate_roadmap(topic)
print(roadmap)

In [ ]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "data visualisation using python"
)